In [ ]:
from langgraph.graph import StateGraph ,START,END
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv




In [ ]:
load_dotenv()

llm = HuggingFaceEndpoint(
     repo_id= "deepseek-ai/DeepSeek-V4-Pro",
        task = "text-generation"
    )

model = ChatHuggingFace(llm=llm)

In [ ]:
class blogstate(TypedDict):
    title: str
    outline:str
    content: str

In [ ]:
def create_outline(state: blogstate) -> blogstate:
    #flatch titel
    title = state['title']
    #create outline based on title
    outline=f'create an outline based on the title: {title}'
    final_outline = model.invoke(outline).content

    #return updated state
    state['outline'] = final_outline
    return state

In [ ]:
def create_blog(state: blogstate) -> blogstate:
    #flatch titel
    title = state['title']
    #create blog based on title and outline
    blog=f'create an blog based on the title: {title} and flow the outline while createing the blog: {state["outline"]}'
    final_blog = model.invoke(blog).content

    #return updated state
    state['content'] = final_blog
    return state

In [ ]:
graph = StateGraph(blogstate)
#defile nodes
graph.add_node("outline",create_outline)
graph.add_node("blog",create_blog)

#define edges
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
#compile graph
workflow = graph.compile()

In [ ]:
input = {"title": "The Future of AI: How Artificial Intelligence is Changing the World",}
final_output = workflow.invoke(input)
print(final_output)

In [ ]:
print("Outline: ", final_output['outline'])